![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `ibm/granite-4-h-small` to analyze car rental customer satisfaction from text

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support of text sentiment analysis in watsonx. It introduces commands for data retrieval, model testing and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.11.


## Learning goal

The goal of this notebook is to demonstrate how to use `ibm/granite-4-h-small` model to analyze customer satisfaction from text.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Data loading](#Data-loading)
3. [Foundation Models on watsonx.ai](#Foundation-Models-on-watsonx.ai)
4. [Analyze the satisfaction](#Analyze-the-satisfaction)
5. [Score the model](#Score-the-model)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install dependencies

In [1]:
%pip install wget | tail -n 1
%pip install "scikit-learn==1.3.2" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Defining the watsonx.ai credentials
This cell defines the watsonx.ai credentials required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">documentation</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Defining the project ID
The Foundation Model requires project ID that provides the context for the call. We will obtain the ID from the project in which this notebook runs. Otherwise, please provide the project ID.

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

### API Client initialization

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id=project_id)

<a id="Data-loading"></a>
## Data loading

Download the `car_rental_training_data` dataset. The dataset provides insight about customers opinions on car rental. It has a label that consists of values: unsatisfied, satisfied.

In [5]:
import pandas as pd
import wget

filename = "car_rental_training_data.csv"
url = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/data/cars-4-you/car_rental_training_data.csv"

if not os.path.isfile(filename):
    wget.download(url, out=filename)

df = pd.read_csv("car_rental_training_data.csv", sep=";")
data = df[["Customer_Service", "Satisfaction"]]

Examine downloaded data.

In [6]:
data.head()

,Customer_Service,Satisfaction
0,I thought the representative handled the initi...,0
1,I have had a few recent rentals that have take...,0
2,car cost more because I didn't pay when I rese...,0
3,I didn't get the car I was told would be avail...,0
4,If there was not a desired vehicle available t...,1


Prepare train and test sets.

In [7]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2)
comments = list(test.Customer_Service)
satisfaction = list(test.Satisfaction)

<a id="Foundation-Models-on-watsonx.ai"></a>
## Foundation Models on watsonx.ai

#### List available models

In [8]:
client.foundation_models.TextModels.show()

{'GRANITE_3_2_8B_INSTRUCT': 'ibm/granite-3-2-8b-instruct', 'GRANITE_3_2B_INSTRUCT': 'ibm/granite-3-2b-instruct', 'GRANITE_3_3_8B_INSTRUCT': 'ibm/granite-3-3-8b-instruct', 'GRANITE_3_8B_INSTRUCT': 'ibm/granite-3-8b-instruct', 'GRANITE_4_H_SMALL': 'ibm/granite-4-h-small', 'GRANITE_8B_CODE_INSTRUCT': 'ibm/granite-8b-code-instruct', 'GRANITE_GUARDIAN_3_8B': 'ibm/granite-guardian-3-8b', 'GRANITE_VISION_3_2_2B': 'ibm/granite-vision-3-2-2b', 'LLAMA_3_2_11B_VISION_INSTRUCT': 'meta-llama/llama-3-2-11b-vision-instruct', 'LLAMA_3_2_90B_VISION_INSTRUCT': 'meta-llama/llama-3-2-90b-vision-instruct', 'LLAMA_3_3_70B_INSTRUCT': 'meta-llama/llama-3-3-70b-instruct', 'LLAMA_3_405B_INSTRUCT': 'meta-llama/llama-3-405b-instruct', 'LLAMA_4_MAVERICK_17B_128E_INSTRUCT_FP8': 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8', 'LLAMA_GUARD_3_11B_VISION': 'meta-llama/llama-guard-3-11b-vision', 'MISTRAL_MEDIUM_2505': 'mistralai/mistral-medium-2505', 'MISTRAL_SMALL_3_1_24B_INSTRUCT_2503': 'mistralai/mistral-small-3

You need to specify `model_id` that will be used for inferencing:

In [9]:
model_id = client.foundation_models.TextModels.GRANITE_4_H_SMALL

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [10]:
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

parameters = {
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 1,
    GenParams.TEMPERATURE: 0,
    GenParams.REPETITION_PENALTY: 1,
}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [11]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(
    model_id=model_id, params=parameters, credentials=credentials, project_id=project_id
)

### Model's details

In [12]:
import json

print(json.dumps(model.get_details(), indent=2))

{
  "model_id": "ibm/granite-4-h-small",
  "label": "granite-4-h-small",
  "provider": "IBM",
  "source": "IBM",
  "indemnity": "IBM_COVERED",
  "functions": [
    {
      "id": "autoai_rag"
    },
    {
      "id": "text_chat"
    },
    {
      "id": "text_generation"
    }
  ],
  "short_description": "Granite-4.0-H-Small is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-Small-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets.",
  "long_description": "Granite-4.0-H-Small is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-Small-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets. This model is developed using a diverse set of techniques with a structured chat format, including supervised finetuning, model alignment using reinforcement learning, and model merging. Granite 4.0 instru

<a id="Analyze-the-satisfaction"></a>
## Analyze the satisfaction

#### Prepare prompt and generate text

In [13]:
instruction = """
Determine if the customer was satisfied with the experience based on the comment.
If the customer should want to continue using the service, otherwise they are not satisfied.
Return simple yes or no.

Comment: The car was broken. They couldn't find a replacement. I've waster over 2 hours.
Satisfied: no

Comment: The service was very good, even though the engine was loud.
Satisfied: yes
"""

In [14]:
prompt1 = "\n".join([instruction, "Comment:" + comments[2], "Satisfied:"])
print(prompt1)


Determine if the customer was satisfied with the experience based on the comment.
If the customer should want to continue using the service, otherwise they are not satisfied.
Return simple yes or no.

Comment: The car was broken. They couldn't find a replacement. I've waster over 2 hours.
Satisfied: no

Comment: The service was very good, even though the engine was loud.
Satisfied: yes

Comment:I would like the personnel to pretend they care about customer, at least.
Satisfied:


Analyze the sentiment for a sample of zero-shot input from the test set.

In [15]:
print(model.generate_text(prompt=prompt1))

 no


### Calculate the accuracy

In [16]:
sample_size = 10
prompts_batch = [
    "\n".join([instruction, "Comment:" + comment, "Satisfied:"])
    for comment in comments[:10]
]
results = [item.strip() for item in model.generate_text(prompt=prompts_batch)]

In [17]:
print(prompts_batch[0])


Determine if the customer was satisfied with the experience based on the comment.
If the customer should want to continue using the service, otherwise they are not satisfied.
Return simple yes or no.

Comment: The car was broken. They couldn't find a replacement. I've waster over 2 hours.
Satisfied: no

Comment: The service was very good, even though the engine was loud.
Satisfied: yes

Comment:The car rental company that I went with had very good customer service. They were out of a certain car I reserved and gave me a upgrade and apologized.
Satisfied:


<a id="Score-the-model"></a>
## Score the model

In [18]:
from sklearn.metrics import accuracy_score

label_map = {0: "no", 1: "yes"}
y_true = [label_map[sat] for sat in satisfaction][:sample_size]

print("accuracy_score", accuracy_score(y_true, results))

accuracy_score 0.9


In [19]:
print("true", y_true, "\npred", results)

true ['yes', 'no', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'yes'] 
pred ['yes', 'no', 'no', 'yes', 'no', 'yes', 'no', 'yes', 'no', 'yes']


<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to analyze car rental customer satisfaction with watsonx.ai foundation model.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Lukasz Cmielowski (Former)**, PhD, Senior Technical Staff Member at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.